# RAG Evaluation — RAGAS

Scores the pipeline on manually created QA pairs.

Requires `bm25_params.json` and a populated Pinecone index from `rag_pipeline.ipynb`.

**Target:** All metrics ≥ 0.8

**Improvement loop:**
- Low context precision → adjust `alpha` or chunk size
- Low context recall → increase `top_k`
- Low faithfulness → tighten the prompt
- Low answer relevancy → tune prompt or retrieval

## Setup — Load credentials

In [ ]:
import os
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(PROJECT_ROOT)
print(f"Working directory: {os.getcwd()}")

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

PINECONE_API_KEY = os.environ["PINECONE_API_KEY"]
OPENAI_API_KEY = os.environ["OPENAI_API_KEY"]
ANTHROPIC_API_KEY = os.environ["ANTHROPIC_API_KEY"]

print("Credentials loaded.")

## Setup — Retriever & Chain

In [ ]:
from pinecone_text.sparse import BM25Encoder
from langchain_openai import OpenAIEmbeddings
from langchain_community.retrievers import PineconeHybridSearchRetriever
from langchain_anthropic import ChatAnthropic
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from pinecone import Pinecone

bm25_loaded = BM25Encoder()
bm25_loaded.load("data/bm25_params.json")

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=OPENAI_API_KEY,
)

pc = Pinecone(api_key=PINECONE_API_KEY)
index = pc.Index("langchain-core-rag")

retriever = PineconeHybridSearchRetriever(
    embeddings=embeddings,
    sparse_encoder=bm25_loaded,
    index=index,
    top_k=5,
    alpha=0.5,
    text_key="text",
)

llm = ChatAnthropic(
    model="claude-sonnet-4-6",
    anthropic_api_key=ANTHROPIC_API_KEY,
)

prompt = ChatPromptTemplate.from_template("""\
You are a code assistant for the LangChain library.
Answer strictly using only the retrieved context below — no outside knowledge.
If the context does not contain enough information, respond with:
\"The information needed to answer this question is not available in the provided context.\"
Do not infer, guess, or make up class names, method names, or code behaviour.

Context:
{context}

Question: {question}
""")

def format_docs(docs):
    return "\n\n".join(
        f"[{d.metadata.get('source', '')}]\n{d.page_content}" for d in docs
    )

chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print("Retriever and chain ready.")

---
## Module 8 — RAGAS Evaluation

Add your 15–20 QA pairs to `test_set` below, then run the cell.

In [ ]:
from datasets import Dataset
from ragas import evaluate
from ragas.metrics import faithfulness, answer_relevancy, context_precision, context_recall

test_set = [
    {
        "question": "What is the purpose of RunnableLambda?",
        "ground_truth": "RunnableLambda wraps a Python callable so it can be used as a Runnable in a LangChain chain.",
    },
    {
        "question": "How does RunnableSequence chain multiple runnables?",
        "ground_truth": "RunnableSequence invokes runnables sequentially where the output of one is the input of the next, constructed via the | operator, .pipe(), or direct instantiation.",
    },
    {
        "question": "What does the ConfigurableField class do?",
        "ground_truth": "ConfigurableField is a NamedTuple that marks a Runnable field as user-configurable at runtime, used with .configurable_fields().",
    },
    # Add more pairs here...
]

eval_data = []
for item in test_set:
    q = item["question"]
    docs = retriever.invoke(q)
    contexts = [d.page_content for d in docs]
    answer = chain.invoke(q)
    eval_data.append({
        "question": q,
        "answer": answer,
        "contexts": contexts,
        "ground_truth": item["ground_truth"],
    })
    print(f"Evaluated: {q[:60]}...")

dataset = Dataset.from_list(eval_data)
results = evaluate(
    dataset,
    metrics=[faithfulness, answer_relevancy, context_precision, context_recall],
)
print("\nResults:")
print(results)